# CLIP / SigLIP 跨模態圖文檢索 (Image-Text Retrieval)

> 模組：`05-Multimodal/02-clip_retrieval`｜2026 版 HuggingFace 繁中 Cookbook

## 本 notebook 的核心一句話

**把「雙塔對比學習」從單模態（文字 vs 文字）升級到跨模態（影像 vs 文字）。**

你在 `02-Adv-tasks` 已經學過用「雙塔模型」把兩段文字各自編碼成向量，再用 cosine 相似度判斷它們是否相關。本節做的事情在心智模型上**完全一樣**，唯一的差別是：其中一座塔吃的是「圖片像素」而非「文字 token」。一旦影像與文字被投影到**同一個共享嵌入空間 (shared embedding space)**，跨模態檢索就退化成你早就會的「向量最近鄰搜尋」。

## 學習目標

1. 理解 CLIP / SigLIP 的對比學習原理：共享嵌入空間、contrastive loss、temperature scaling。
2. 用 `AutoProcessor` 同時前處理「文字」與「影像」兩種模態。
3. 批次計算 `image_embeds` / `text_embeds`，並做 L2 normalize。
4. 實作 **zero-shot 影像分類**（不需任何訓練資料）。
5. 用 FAISS 建立影像索引，做 **text-to-image** 與 **image-to-text** 雙向檢索。
6. 用 **Recall@k / MRR** 量化檢索品質。
7. 比較 **SigLIP (sigmoid loss) vs CLIP (softmax loss)**，以及中文檢索的多語模型選擇。

## 前置知識（建議先讀過）

- 雙塔 / cosine embedding 對比學習：[`../../02-Adv-tasks/04-sentence_similarity/dual_model.ipynb`](../../02-Adv-tasks/04-sentence_similarity/dual_model.ipynb)
- FAISS 向量檢索：[`../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`](../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb)
- HuggingFace 元件基礎（Processor / Dataset）：[`../../01-Component/`](../../01-Component/)

## 銜接

本節是**多模態 embedding 與多模態 RAG 的理論基礎**。下一節會把這裡學到的「圖文共享嵌入」接上向量資料庫，組成可問圖、可問文的多模態 RAG。

## Step 0 環境與版本鎖定

**WHY**：2026 全 repo 統一鎖版本，避免 `transformers` / `datasets` API 漂移造成範例失效。CLIP/SigLIP 屬於相對輕量的 encoder（base 版約數百 MB），CPU 也能跑 demo，但有 GPU 會快很多。FAISS 用 `faiss-cpu` 即可，本節的向量規模（數百到數千張圖）完全不需要 GPU 版 FAISS。

> 注意：這裡的 `pip install` 與既有 notebook 一樣只在第一次執行時需要；若你的環境已備妥可跳過。

In [ ]:
# Pinned versions for 2026 cookbook consistency.
# SigLIP2 requires transformers >= 4.46; siglip/clip processors live in transformers core.
!pip install -q "transformers>=4.46" "datasets>=3.0" "accelerate>=1.0" "evaluate>=0.4"
!pip install -q faiss-cpu pillow matplotlib
# Optional: jina-clip-v2 (multilingual / Chinese-friendly) needs einops + timm.
!pip install -q einops timm

In [ ]:
import torch
import numpy as np
from PIL import Image

# Reproducibility: same seed convention as the rest of the cookbook.
torch.manual_seed(42)
np.random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 only makes sense on GPU; keep float32 on CPU for numerical stability.
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print(f"device={DEVICE}, dtype={DTYPE}")

## Step 1 原理：共享嵌入空間與對比學習

### 1.1 從雙塔文字相似度說起（你已經會的部分）

在 `dual_model.ipynb` 裡，你做的是：

- 塔 A：`encoder(sentence1) -> vec_a`
- 塔 B：`encoder(sentence2) -> vec_b`
- 損失：`CosineEmbeddingLoss`，正樣本拉近 `cos(vec_a, vec_b) -> 1`，負樣本推遠 `-> -1`。

關鍵直覺：**「相關」= 向量在同一空間中靠近**。

### 1.2 CLIP 把「文字塔 + 文字塔」換成「影像塔 + 文字塔」

CLIP (Contrastive Language-Image Pre-training) 只改了一件事：

| 元件 | 雙塔文字模型 | CLIP |
| :-- | :-- | :-- |
| 塔 A | Text Encoder | **Image Encoder** (ViT) |
| 塔 B | Text Encoder | Text Encoder |
| 投影 | 線性層到共享維度 | 線性層到共享維度 |
| 相似度 | cosine | cosine |
| 損失 | CosineEmbeddingLoss | **InfoNCE / 對比 softmax** |

因為兩座塔的輸出落在**同一個 D 維空間**，所以「圖片 vec」可以直接跟「文字 vec」算 cosine。這就是「跨模態」的全部魔法。

### 1.3 對比損失與 temperature scaling

訓練時一個 batch 有 N 對 (image, text)。CLIP 算出 N×N 的相似度矩陣，對角線是正配對，其餘是負配對。損失是「讓對角線在每一列/每一行的 softmax 中勝出」：

```
logits = (image_embeds @ text_embeds.T) / temperature   # 縮放
loss   = (cross_entropy(logits, labels)            # image->text 方向
        + cross_entropy(logits.T, labels)) / 2     # text->image 方向
```

`temperature`（CLIP 實作成可學習的 `logit_scale`）控制 softmax 的銳利度：溫度越低，模型對「最相似」越自信。這正是 `CosineEmbeddingLoss` 的「margin」在對比學習裡的對應物——只是這裡用 batch 內所有負樣本一次比較，比單純正負對更有效率。

### 1.4 SigLIP 的差異（第 9 步會深入）

SigLIP 把「整列 softmax」換成「每個配對獨立的 sigmoid 二分類」。損失從「在 batch 中選一個對的」變成「每一對自己判斷對不對」，因此**不需要大 batch 做全域正規化**，訓練更穩、小 batch 也好。SigLIP2 (2026 首選) 再加上多語、更強的視覺特徵與動態解析度支援。

## Step 2 載入模型與 `AutoProcessor`

**WHY**：在單模態你用 `AutoTokenizer` 處理文字。多模態的對應物是 `AutoProcessor`——它是一個「複合前處理器」，內部同時包了 `tokenizer`（處理文字 -> `input_ids`）與 `image_processor`（處理圖片 -> `pixel_values`：resize、center crop、normalize）。一個 processor 同時吐出兩種模態的 tensor，這是多模態相對單模態最重要的 API 差異。

我們首選 **SigLIP2**（2026 推薦），並準備好 CLIP 作為經典對照（第 9 步會兩者對打）。

> VRAM 提示：`siglip2-base` / `clip-vit-base` 皆為輕量級（< 1GB），不需要量化。若要追求最高檢索品質才考慮 `laion/CLIP-ViT-H-14`（約 4GB，建議 GPU + bf16）。本 notebook 為教學用途維持 base 版即可。

In [ ]:
from transformers import AutoProcessor, AutoModel

# --- 2026 首選：SigLIP2 (multilingual, sigmoid contrastive) ---
SIGLIP_ID = "google/siglip2-base-patch16-224"

# --- 經典對照：原始 OpenAI CLIP ---
CLIP_ID = "openai/clip-vit-base-patch32"

# Load the primary model (SigLIP2). AutoModel returns the dual-tower model
# exposing get_image_features() / get_text_features().
processor = AutoProcessor.from_pretrained(SIGLIP_ID)
model = AutoModel.from_pretrained(SIGLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()

print(type(model).__name__)
print("processor packs:", type(processor).__name__)

### 2.1 看清 processor 同時吐出兩種模態

**WHY**：在動真格之前，先用一張假圖 + 一句文字看看 processor 的輸出長相，建立「文字 -> `input_ids`、影像 -> `pixel_values`」的肌肉記憶。這一步在單模態版本你只會看到 `input_ids` / `attention_mask`，現在多了 `pixel_values`。

In [ ]:
# Build one throwaway sample to inspect the dual-modality output shapes.
dummy_img = Image.new("RGB", (256, 256), color=(180, 120, 60))
dummy_text = ["一隻貓", "一輛紅色跑車"]

# SigLIP convention: padding="max_length" (it was pretrained with fixed-length text).
batch = processor(
    text=dummy_text,
    images=dummy_img,
    padding="max_length",
    return_tensors="pt",
)

for k, v in batch.items():
    print(f"{k:>16}: {tuple(v.shape)}")
# Expect input_ids (text tower) and pixel_values (image tower) side by side.

## Step 3 載入小型圖文資料集

**WHY**：要做檢索就需要一個「圖庫 + 對應描述」的小資料集。我們用一個常見的圖文 caption 資料集（Flickr 風格）取一小批當作 demo 圖庫。我們刻意只取數十筆，讓 CPU 也能在合理時間跑完——教學重點是流程，不是規模。

資料的核心結構就兩欄：

- `image`：PIL 影像（影像塔的輸入）
- `caption`：文字描述（文字塔的輸入）

> 自備中文商品圖文：若你有電商商品圖，只要整理成 `{image, caption}` 兩欄（caption 用中文商品名/描述），後續所有程式碼都不必改。下一個 cell 也示範了如何用本機資料夾自建 dataset。

In [ ]:
from datasets import load_dataset

N_GALLERY = 40  # keep tiny for CPU-friendly demo; bump up if you have a GPU.

# A small, widely-mirrored image-caption dataset. Streaming avoids a full download.
try:
    stream = load_dataset("nlphuji/flickr30k", split="test", streaming=True)
    rows = []
    for ex in stream:
        # flickr30k stores a list of captions per image; take the first one.
        cap = ex["caption"][0] if isinstance(ex["caption"], list) else ex["caption"]
        rows.append({"image": ex["image"].convert("RGB"), "caption": cap})
        if len(rows) >= N_GALLERY:
            break
    print(f"loaded {len(rows)} image-caption pairs")
except Exception as e:
    print("dataset download failed, falling back to synthetic demo:", e)
    rows = None

### 3.1 後備方案 / 自備資料：從本機資料夾建圖庫

**WHY**：教學環境常常連不上 HuggingFace Hub，或你想直接用自己的中文商品圖。下面這段示範「只要有一個資料夾的圖片」就能組成同樣的 `{image, caption}` 結構，後續程式碼完全共用。若上一個 cell 成功，這段會被跳過。

In [ ]:
from pathlib import Path

if rows is None:
    # Point this at your own folder of product images, e.g. ./my_images/*.jpg
    img_dir = Path("./my_images")
    rows = []
    if img_dir.exists():
        for p in sorted(img_dir.glob("*.[jp][pn]g"))[:N_GALLERY]:
            rows.append({
                "image": Image.open(p).convert("RGB"),
                # Use filename stem as a placeholder caption; replace with real text.
                "caption": p.stem,
            })
    if not rows:
        # Last-resort synthetic gallery so the notebook always runs end-to-end.
        palette = [(220, 30, 30), (30, 160, 60), (40, 80, 220), (230, 200, 20)]
        names = ["紅色方塊", "綠色方塊", "藍色方塊", "黃色方塊"]
        rows = [
            {"image": Image.new("RGB", (224, 224), c), "caption": f"一張{n}的照片"}
            for c, n in zip(palette, names)
        ]
    print(f"using {len(rows)} fallback pairs")

images = [r["image"] for r in rows]
captions = [r["caption"] for r in rows]
print("sample caption:", captions[0])

## Step 4 批次計算 embeddings 並 L2 normalize

**WHY**：這是整個 notebook 的心臟，也是跟雙塔文字模型**最像**的一步。

- 影像塔：`model.get_image_features(pixel_values)` -> `image_embeds`
- 文字塔：`model.get_text_features(input_ids)` -> `text_embeds`

兩者輸出維度相同（共享空間）。然後做 **L2 normalize**——這一步在 `retrieval_bot.ipynb` 你已經做過，原因不變：normalize 之後「內積 = cosine 相似度」，於是我們能用 FAISS 的 `IndexFlatIP`（內積索引）直接當 cosine 檢索器用。

> 為什麼分批：圖庫可能很大，一次塞進 GPU 會 OOM。用小批次累積，跟你訓練時的 dataloader 思路一致。`torch.no_grad()` 關掉梯度省記憶體（純推論）。

In [ ]:
@torch.no_grad()
def encode_images(pil_images, batch_size=8):
    """Encode a list of PIL images into L2-normalized embeddings (numpy float32)."""
    out = []
    for i in range(0, len(pil_images), batch_size):
        chunk = pil_images[i : i + batch_size]
        inputs = processor(images=chunk, return_tensors="pt").to(DEVICE)
        feats = model.get_image_features(**inputs)
        feats = torch.nn.functional.normalize(feats, p=2, dim=-1)  # cosine-ready
        out.append(feats.float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def encode_texts(texts, batch_size=16):
    """Encode a list of strings into L2-normalized embeddings (numpy float32)."""
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        # SigLIP was trained with fixed-length text -> padding="max_length".
        inputs = processor(
            text=chunk, padding="max_length", return_tensors="pt"
        ).to(DEVICE)
        feats = model.get_text_features(**inputs)
        feats = torch.nn.functional.normalize(feats, p=2, dim=-1)
        out.append(feats.float().cpu())
    return torch.cat(out).numpy()

In [ ]:
# Encode the whole gallery once. These are the vectors we will index & search.
image_embeds = encode_images(images)
text_embeds = encode_texts(captions)

print("image_embeds:", image_embeds.shape, image_embeds.dtype)
print("text_embeds :", text_embeds.shape)
# Sanity check: norms should all be ~1.0 after L2 normalization.
print("image norm[0]:", round(float(np.linalg.norm(image_embeds[0])), 4))

## Step 5 Zero-shot 影像分類

**WHY**：CLIP 最著名的能力是「不訓練就能分類」。原理仍然是共享空間裡的 cosine：把每個候選類別包成一句 prompt（例如 `「一張貓的照片」`），編成文字向量；再把待分類圖片編成影像向量；圖片屬於「跟它 cosine 最高的那個類別」。

這裡有兩個關鍵實作細節：

1. **Prompt 模板**：CLIP 對 prompt 措辭敏感。用 `「一張{label}的照片」` 這種完整句子，比單個詞效果好（這叫 prompt engineering，跟 LLM 的同名技巧同源）。
2. **`logits_per_image`**：HuggingFace 的 CLIP/SigLIP forward 會直接回傳 `logits_per_image`（= 影像對每個文字的縮放相似度）。CLIP 對它做 `softmax` 得機率；SigLIP 對它做 `sigmoid`（各類別獨立機率）。這正是兩者損失差異在推論端的體現。

In [ ]:
# Define candidate labels and wrap each in a natural-language prompt template.
candidate_labels = ["貓", "狗", "汽車", "飛機", "人", "建築物", "食物", "風景"]
prompts = [f"一張{lab}的照片" for lab in candidate_labels]  # prompt template

query_image = images[0]  # classify the first gallery image as a demo

with torch.no_grad():
    inputs = processor(
        text=prompts, images=query_image, padding="max_length", return_tensors="pt"
    ).to(DEVICE)
    outputs = model(**inputs)
    # logits_per_image: shape (num_images=1, num_labels)
    logits = outputs.logits_per_image

    # SigLIP uses sigmoid (independent per-class prob); CLIP uses softmax.
    probs = torch.sigmoid(logits)  # for CLIP, swap to logits.softmax(dim=-1)

probs = probs[0].float().cpu().numpy()
for lab, p in sorted(zip(candidate_labels, probs), key=lambda x: -x[1]):
    print(f"{lab:>6}: {p:.3f}")

## Step 6 建立 FAISS 影像索引：text-to-image 檢索

**WHY**：到這裡，跨模態檢索已經跟 `retrieval_bot.ipynb` 的 FAQ 檢索**一模一樣**——差別只在「索引裡放的是影像向量，query 是文字向量」。因為兩者在同一空間，FAISS 不在乎向量原本來自像素還是 token。

- `IndexFlatIP`：內積索引。因為我們已 L2 normalize，內積即 cosine，分數越高越相似。
- 流程：`index.add(image_embeds)` 建庫 -> 用文字 query 編碼 -> `index.search(query_vec, k)` 取回最相似的圖片 id。

這就是「以文搜圖」：使用者打一句話，系統回傳最符合的圖片。

In [ ]:
import faiss

DIM = image_embeds.shape[1]
index = faiss.IndexFlatIP(DIM)  # inner product == cosine on normalized vectors
index.add(image_embeds.astype(np.float32))  # gallery = all images
print("indexed vectors:", index.ntotal)


def text_to_image(query: str, k: int = 5):
    """Return (gallery_index, score) for the top-k images matching a text query."""
    q = encode_texts([query]).astype(np.float32)  # same space as image_embeds
    scores, idx = index.search(q, k)
    return list(zip(idx[0].tolist(), scores[0].tolist()))


# Demo query. Use a description; the index returns the closest images.
results = text_to_image("一個人在戶外運動", k=3)
for rank, (i, s) in enumerate(results, 1):
    print(f"#{rank}  score={s:.3f}  caption={captions[i]!r}")

### 6.1 視覺化檢索結果

**WHY**：檢索系統一定要肉眼驗證，數字漂亮不代表抓對圖。畫出 top-k 圖片與分數，是上線前的基本 sanity check。

In [ ]:
import matplotlib.pyplot as plt


def show_text_to_image(query: str, k: int = 3):
    hits = text_to_image(query, k)
    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
    if k == 1:
        axes = [axes]
    for ax, (i, s) in zip(axes, hits):
        ax.imshow(images[i])
        ax.set_title(f"score={s:.3f}")
        ax.axis("off")
    fig.suptitle(f"query: {query}")
    plt.tight_layout()
    plt.show()


show_text_to_image("一個人在戶外運動", k=3)

## Step 7 反向：image-to-text 檢索（以圖搜文）

**WHY**：共享空間天生對稱。前一步我們把影像放進 FAISS、用文字查；現在反過來——把**文字描述向量**放進 FAISS、用**影像向量**查，就得到「給一張圖，找最貼切的文字描述」。

這正是雙塔的優雅之處：同一組 embedding，換一下誰當索引、誰當 query，就得到反方向的能力。實務應用包括：自動圖片標註、商品圖配對最相關的文案。

In [ ]:
# Build a second index over caption (text) embeddings.
text_index = faiss.IndexFlatIP(DIM)
text_index.add(text_embeds.astype(np.float32))


def image_to_text(pil_image, k: int = 5):
    """Return (caption_index, score) for the top-k captions matching an image."""
    q = encode_images([pil_image]).astype(np.float32)
    scores, idx = text_index.search(q, k)
    return list(zip(idx[0].tolist(), scores[0].tolist()))


# Take a gallery image and find its best-matching captions.
probe_i = 0
for rank, (j, s) in enumerate(image_to_text(images[probe_i], k=3), 1):
    print(f"#{rank}  score={s:.3f}  caption={captions[j]!r}")
print("\nground-truth caption:", captions[probe_i])

## Step 8 檢索評測：Recall@k 與 MRR

**WHY**：到目前都是肉眼判斷，要上線就得量化。圖文 caption 資料集天然提供 ground truth：第 i 張圖的正確描述就是第 i 條 caption。於是可以系統性地問：「用 caption i 去查圖，正確的圖 i 有沒有排進前 k 名？」

兩個標準檢索指標（補上 inventory 缺的部分）：

- **Recall@k**：正確答案出現在前 k 名的比例。Recall@1 是嚴格命中率，Recall@5 容忍排序誤差。
- **MRR (Mean Reciprocal Rank)**：正確答案排名倒數的平均。排第 1 給 1.0、排第 2 給 0.5……綜合反映「正確答案排多前面」。

這兩個指標跟你在文字檢索學的定義完全相同——再次印證跨模態只是換了向量來源。

In [ ]:
def evaluate_retrieval(query_embeds, gallery_index, ks=(1, 5, 10)):
    """Diagonal ground truth: query i should retrieve gallery item i.

    Returns dict with Recall@k for each k and MRR.
    """
    n = query_embeds.shape[0]
    max_k = max(ks)
    scores, idx = gallery_index.search(query_embeds.astype(np.float32), max_k)

    recall = {k: 0 for k in ks}
    rr_sum = 0.0
    for i in range(n):
        retrieved = idx[i].tolist()
        if i in retrieved:
            rank = retrieved.index(i) + 1  # 1-based rank of the correct item
            rr_sum += 1.0 / rank
            for k in ks:
                if rank <= k:
                    recall[k] += 1
    metrics = {f"Recall@{k}": recall[k] / n for k in ks}
    metrics["MRR"] = rr_sum / n
    return metrics


# text-to-image: query with caption vectors against the image index.
t2i = evaluate_retrieval(text_embeds, index)
# image-to-text: query with image vectors against the caption index.
i2t = evaluate_retrieval(image_embeds, text_index)

print("text -> image:", {k: round(v, 3) for k, v in t2i.items()})
print("image -> text:", {k: round(v, 3) for k, v in i2t.items()})

### 8.1 解讀

- 在這種小型乾淨資料集上，base 版 SigLIP/CLIP 通常 Recall@5 很高（接近 1.0）；難度在 Recall@1，因為很多圖的描述彼此相近（多張運動照、多張街景）。
- 若你換成**中文** caption 而 Recall 明顯下滑，那不是程式碼錯，而是**模型語言能力**問題——原始 OpenAI CLIP 幾乎只懂英文。這把我們帶到第 9 步的模型選擇。

## Step 9 SigLIP vs CLIP，以及中文檢索的模型選擇

### 9.1 損失函數差異：softmax (CLIP) vs sigmoid (SigLIP)

| 面向 | CLIP (softmax / InfoNCE) | SigLIP (sigmoid) |
| :-- | :-- | :-- |
| 正規化 | 在整個 batch 上做 softmax（需全域比較） | 每對獨立 sigmoid（無全域耦合） |
| Batch 依賴 | 需要**大 batch** 才有足夠負樣本 | 小 batch 也穩定 |
| 損失語意 | 「在這 N 個裡選對的那個」 | 「這一對到底配不配」 |
| 推論機率 | `logits.softmax(-1)`（類別互斥） | `logits.sigmoid()`（類別獨立） |
| 多標籤友善 | 較弱（強制互斥） | 較佳（可同時多個高分） |

直覺對應你學過的東西：CLIP 的 softmax 像「多選一分類」，SigLIP 的 sigmoid 像「多個獨立二分類」。後者讓「一張圖同時是『戶外』又是『運動』」這種情況更自然。

### 9.2 中文 / 多語檢索怎麼選

- `openai/clip-vit-base-patch32`：經典基準，但**只懂英文**，中文 query 幾乎不可用。
- `google/siglip2-base-patch16-224`：2026 首選，**原生多語**訓練，中文檢索可用且品質好。
- `jinaai/jina-clip-v2`：明確標榜多語（含中文）與長文本，商品文案/中文 caption 場景很合適。
- `laion/CLIP-ViT-H-14-laion2B-s32B-b79K`：英文檢索品質天花板，但約 4GB，需 GPU + bf16，且仍以英文為主。

下面用一句**中文** query 對 SigLIP 與 CLIP 做對照，直接感受語言能力差距。

In [ ]:
# Load the classic CLIP for a side-by-side comparison.
clip_processor = AutoProcessor.from_pretrained(CLIP_ID)
clip_model = AutoModel.from_pretrained(CLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()


@torch.no_grad()
def clip_encode_images(pil_images, batch_size=8):
    out = []
    for i in range(0, len(pil_images), batch_size):
        inputs = clip_processor(
            images=pil_images[i : i + batch_size], return_tensors="pt"
        ).to(DEVICE)
        f = clip_model.get_image_features(**inputs)
        f = torch.nn.functional.normalize(f, p=2, dim=-1)
        out.append(f.float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def clip_encode_texts(texts):
    # CLIP uses dynamic padding (not max_length like SigLIP).
    inputs = clip_processor(
        text=texts, padding=True, truncation=True, return_tensors="pt"
    ).to(DEVICE)
    f = clip_model.get_text_features(**inputs)
    f = torch.nn.functional.normalize(f, p=2, dim=-1)
    return f.float().cpu().numpy()

In [ ]:
# Same Chinese query against both models; compare top-1 retrieved caption.
zh_query = "一個人在戶外運動"

# SigLIP (multilingual) path -- reuse the already-built image index.
sig_top = text_to_image(zh_query, k=1)[0]
print(f"[SigLIP2] top-1 score={sig_top[1]:.3f} caption={captions[sig_top[0]]!r}")

# CLIP path: build a temporary CLIP image index, query with CLIP text vector.
clip_img_embeds = clip_encode_images(images)
clip_index = faiss.IndexFlatIP(clip_img_embeds.shape[1])
clip_index.add(clip_img_embeds.astype(np.float32))
qv = clip_encode_texts([zh_query]).astype(np.float32)
cs, ci = clip_index.search(qv, 1)
print(f"[CLIP   ] top-1 score={cs[0][0]:.3f} caption={captions[ci[0][0]]!r}")

# Note: scores across models are NOT directly comparable (different spaces).
# What matters is whether each model retrieves a *sensible* image for the query.

### 9.3 觀察重點

- **不要直接比兩個模型的分數絕對值**——它們在各自的嵌入空間、用各自的 `logit_scale`，數值不可跨模型比較。要比的是「抓回來的圖合不合理」。
- 對英文 query，CLIP 通常表現很好；對中文 query，CLIP 常常抓回不相關的圖，而 SigLIP2 / jina-clip-v2 仍穩定。這就是多語模型在地化檢索的價值。
- 想用 `jina-clip-v2`：把 `SIGLIP_ID` 換成 `"jinaai/jina-clip-v2"` 並在 `from_pretrained` 加 `trust_remote_code=True`（它帶自訂模型碼）即可，其餘流程不變。

## Step 10 小結：嵌入即介面

### 你完成了什麼

1. 把雙塔對比學習的心智模型從「文字 vs 文字」延伸到「影像 vs 文字」，理解了共享嵌入空間、contrastive loss 與 temperature scaling 如何銜接你已熟悉的 `CosineEmbeddingLoss`。
2. 用一個 `AutoProcessor` 同時前處理兩種模態（`input_ids` + `pixel_values`）。
3. 批次編碼 + L2 normalize，讓 FAISS `IndexFlatIP` 直接當 cosine 檢索器。
4. 實作了 zero-shot 分類、text-to-image、image-to-text 三種能力——全部共用同一組 embedding。
5. 用 Recall@k / MRR 量化檢索品質，並比較了 SigLIP (sigmoid) 與 CLIP (softmax) 及中文模型選擇。

### 核心洞察：嵌入即介面 (Embeddings as the Interface)

一旦不同模態被投影到**同一個向量空間**，模態之間的界線就消失了。檢索、分類、配對都退化成「向量最近鄰」這一個操作。這也是為什麼向量資料庫成為多模態系統的通用底座——**它不在乎向量來自文字、像素還是音訊**。

### 練習題

1. **prompt 工程**：把第 5 步的模板從 `「一張{label}的照片」` 改成單詞 `「{label}」`，觀察 zero-shot 準確率變化，並解釋為什麼完整句子通常較好。
2. **規模化**：把 `N_GALLERY` 調到 500（建議 GPU），重新計算 Recall@1 / Recall@5，思考「圖庫變大時 Recall 為何下降、MRR 為何更能反映排序品質」。
3. **多語對照**：載入 `jinaai/jina-clip-v2`（`trust_remote_code=True`），用同一組中文 query 跑第 8 步評測，與 SigLIP2 比較 Recall@5。
4. **索引升級**：把 `IndexFlatIP`（暴力搜尋）換成 `IndexIVFFlat`（需 `train()`），比較大圖庫下的查詢延遲與召回取捨——這是 `retrieval_bot.ipynb` 提過的近似最近鄰主題。
5. **去重應用**：用 image-to-image（影像查影像）找出圖庫中視覺上重複的圖片，思考電商選品的應用。

### 通往下一節

本節證明了「圖文可以共享一個向量空間並互相檢索」。下一節把這個能力接上生成模型，組成**多模態 RAG**：使用者用文字或圖片發問 -> 用本節的跨模態檢索取回相關圖文證據 -> 餵給多模態 LLM (VLM) 生成有依據的回答。你在這裡建立的 embedding 與 FAISS 流程，就是那套 RAG 的檢索骨幹。

> 銜接閱讀：回顧 [`../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`](../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb) 的 FAISS 檢索流程，對照本節的跨模態版本，你會發現兩者只差在「向量從哪來」。